In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import data_utils as du
import preprocessing as pp
import evaluation as ev
import baselines as bl
(ROOT / "results" / "baselines").mkdir(parents=True, exist_ok=True)

## Carga y Preparación de los datos

In [2]:
splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits) # Contenido de train/dev

for df in splits.values():
    df["input"] = pp.prepare(df, media_re)
    df["input_sin_titular"] = pp.prepare(df, media_re, with_headline=False)
    df["input_raw"] = [pp.model_input(h, t) for h, t in zip(df["Headline"], df["Text"])]
tr, dev, test = splits["train"], splits["dev"], splits["test"]
print({k: (len(v), v.label.mean().round(3)) for k, v in splits.items()})

{'train': (676, np.float64(0.5)), 'dev': (290, np.float64(0.479)), 'test': (572, np.float64(0.5))}


## Evaluación común y baselines triviales

`fit_eval` entrena en train y evalúa en dev y test con dos reglas de decisión: umbral base (0,5; 0 para el margen del SVM) y rango (se marca como Fake el 50 % mejor puntuado). Con los `dummy` solo se usa el umbral: sus puntuaciones están empatadas.

In [ ]:
def fit_eval(name, kind, clf, col="input", C=1.0, train=tr):
    """Entrena en `train` y evalúa en dev y test con umbral base y con decisión por rango (50 %)."""
    m = bl.make_pipe(kind, C, clf).fit(train[col], train.label)
    rows, thr = [], (0.0 if clf == "svm" else 0.5)   # el margen del SVM se corta en 0
    for split, df in (("dev", dev), ("test", test)):
        s = bl.get_scores(m, df[col])
        decisiones = [("umbral", ev.predict_threshold(s, thr))]
        if not clf.startswith("dummy"):               
            decisiones.append(("rango50", ev.predict_by_rank(s, 0.5)))
        for dec, pred in decisiones:
            rows.append({**ev.evaluate(df.label, pred, s, name), "split": split, "decision": dec})
    return rows


def mostrar(res, split):
    cols = ["modelo", "decision", "f1_fake", "precision", "recall", "pct_pred_fake", "auc"]
    return res[res.split == split][cols].reset_index(drop=True)

In [4]:
rows = []
for clf in ("dummy_fake", "dummy_true"):
    rows += fit_eval(clf, "word", clf)
    
    

## Clasificadores TF-IDF

Palabras y caracteres × regresión logística, SVM lineal y ComplementNB. Se comparan F1 Fake, AUC y, sobre todo, `pct_pred_fake` frente al 50 % real.

In [5]:
for kind in ("word", "char"):
    for clf in ("lr", "svm", "cnb"):
        rows += fit_eval(f"{kind}-{clf}", kind, clf)
res = ev.results_table(rows)
print("DEV");  display(mostrar(res, "dev"))
print("TEST"); display(mostrar(res, "test"))

DEV


,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,char-svm,umbral,0.806,0.792,0.820,0.497,0.886
1,char-svm,rango50,0.803,0.786,0.820,0.500,0.886
2,word-svm,umbral,0.793,0.801,0.784,0.469,0.888
3,word-cnb,rango50,0.789,0.772,0.806,0.500,0.858
4,char-lr,umbral,0.782,0.766,0.799,0.500,0.868
5,word-lr,rango50,0.782,0.766,0.799,0.500,0.873
6,word-svm,rango50,0.782,0.766,0.799,0.500,0.888
7,char-lr,rango50,0.782,0.766,0.799,0.500,0.868
8,word-lr,umbral,0.771,0.766,0.777,0.486,0.873
9,char-cnb,rango50,0.768,0.752,0.784,0.500,0.847


TEST


,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,word-svm,rango50,0.741,0.741,0.741,0.500,0.792
1,char-svm,rango50,0.741,0.741,0.741,0.500,0.812
2,char-lr,rango50,0.724,0.724,0.724,0.500,0.798
3,char-cnb,rango50,0.717,0.717,0.717,0.500,0.780
4,word-lr,rango50,0.706,0.706,0.706,0.500,0.779
5,word-cnb,rango50,0.689,0.689,0.689,0.500,0.753
6,dummy_fake,umbral,0.667,0.500,1.000,1.000,0.500
7,char-svm,umbral,0.655,0.829,0.542,0.327,0.812
8,char-lr,umbral,0.647,0.834,0.528,0.316,0.798
9,word-lr,umbral,0.646,0.799,0.542,0.339,0.779


## Análisis de contribución: titular y limpieza

`input` (limpio, con titular), `input_sin_titular` (limpio, solo cuerpo) e `input_raw` (sin limpiar), con regresión logística y SVM.

In [6]:
abl = []
for nombre, kind, clf in (("char-lr", "char", "lr"), ("char-svm", "char", "svm")):
    for col in ("input", "input_sin_titular", "input_raw"):
        abl += fit_eval(f"{nombre} [{col}]", kind, clf, col=col)
rows += abl
ab = ev.results_table(abl)
print("DEV");  display(mostrar(ab, "dev"))
print("TEST"); display(mostrar(ab, "test"))

DEV


,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,char-svm [input_raw],umbral,0.807,0.788,0.827,0.503,0.886
1,char-svm [input],umbral,0.806,0.792,0.820,0.497,0.886
2,char-svm [input],rango50,0.803,0.786,0.820,0.500,0.886
3,char-svm [input_sin_titular],rango50,0.803,0.786,0.820,0.500,0.888
4,char-svm [input_raw],rango50,0.803,0.786,0.820,0.500,0.886
5,char-svm [input_sin_titular],umbral,0.801,0.790,0.813,0.493,0.888
6,char-lr [input],umbral,0.782,0.766,0.799,0.500,0.868
7,char-lr [input],rango50,0.782,0.766,0.799,0.500,0.868
8,char-lr [input_raw],rango50,0.782,0.766,0.799,0.500,0.869
9,char-lr [input_sin_titular],umbral,0.779,0.760,0.799,0.503,0.871


TEST


,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,char-svm [input_raw],rango50,0.741,0.741,0.741,0.500,0.810
1,char-svm [input_sin_titular],rango50,0.741,0.741,0.741,0.500,0.811
2,char-svm [input],rango50,0.741,0.741,0.741,0.500,0.812
3,char-svm [input_raw],umbral,0.731,0.773,0.692,0.448,0.810
4,char-lr [input_raw],rango50,0.727,0.727,0.727,0.500,0.796
5,char-lr [input_sin_titular],rango50,0.724,0.724,0.724,0.500,0.796
6,char-lr [input],rango50,0.724,0.724,0.724,0.500,0.798
7,char-lr [input_raw],umbral,0.693,0.741,0.650,0.439,0.796
8,char-svm [input_sin_titular],umbral,0.662,0.835,0.549,0.329,0.811
9,char-svm [input],umbral,0.655,0.829,0.542,0.327,0.812


## Validación cruzada por grupos (solo train)

`GroupKFold` con los grupos de `results/eda/10_train_grupos.csv`.

In [7]:
# Validación cruzada por grupos

groups = du.group_ids(ROOT, tr)
cv_rows = []
for kind, clf in (("word", "lr"), ("char","lr"), ("char", "svm")):
    s = bl.cv_scores(bl.make_pipe(kind, 1.0, clf), tr["input"], tr.label, groups)
    thr = 0.0 if clf == "svm" else 0.5
    cv_rows.append(ev.evaluate(tr.label, ev.predict_threshold(s, thr), s, f"CV {kind}-{clf}"))
ev.results_table(cv_rows)

,modelo,f1_fake,precision,recall,pct_pred_fake,pct_real_fake,n,auc
0,CV char-svm,0.850,0.845,0.855,0.506,0.5,676,0.922
1,CV word-lr,0.825,0.827,0.822,0.497,0.5,676,0.896
2,CV char-lr,0.824,0.799,0.849,0.531,0.5,676,0.898


## Leave-topic-out

Dejar fuera un tema cada vez. Mide la sensibilidad al tema, no el cambio de dominio real (COVID, otros países).

In [8]:
# Leave-topic-out sobre train + dev (966 noticias). Topic solo sirve para partir, nunca como característica
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
td = pd.concat([tr, dev], ignore_index=True)
s = cross_val_predict(bl.make_pipe("char", 1.0, "lr"), td["input"], td.label,
                      groups=td["Topic"], cv=LeaveOneGroupOut(), method="predict_proba")[:, 1]
td["s"] = s
pd.DataFrame([{**ev.evaluate(g.label, ev.predict_threshold(g.s), g.s, t), "topic": t}
              for t, g in td.groupby("Topic")]).round(3)

,modelo,f1_fake,precision,recall,pct_pred_fake,pct_real_fake,n,auc,topic
0,Economy,0.848,1.000,0.737,0.326,0.442,43,0.897,Economy
1,Education,0.917,0.917,0.917,0.545,0.545,22,0.917,Education
2,Entertainment,0.785,0.664,0.962,0.764,0.527,148,0.903,Entertainment
3,Health,0.732,0.833,0.652,0.391,0.500,46,0.894,Health
4,Politics,0.801,0.762,0.845,0.511,0.461,321,0.906,Politics
5,Science,0.800,0.842,0.762,0.432,0.477,88,0.900,Science
6,Security,0.791,0.944,0.680,0.429,0.595,42,0.880,Security
7,Society,0.861,0.922,0.808,0.481,0.549,133,0.938,Society
8,Sport,0.771,0.651,0.947,0.675,0.463,123,0.892,Sport


## Lectura de resultados (dev, validación cruzada y leave-topic-out)

- **Dev:** los clasificadores lineales quedan entre 0,77 y 0,81 de F1 Fake (AUC 0,86-0,89). Con 290 noticias la diferencia entre ellos (~0,02-0,03) está dentro del ruido; char-svm es el mejor, sin ventaja clara. El suelo útil es `dummy_fake` (0,648).

- **ComplementNB** depende mucho de la regla de decisión (char-cnb: 0,768 con rango frente a 0,597 con umbral; predice un 25 % de Fake): sus probabilidades están mal calibradas. No se usa en las comparaciones de umbral.
- **Validación cruzada en train:** F1 0,82-0,85 y AUC 0,90-0,92, **por encima de dev**. Es optimista (train comparte medios y estilo entre pliegues); sirve para comparar modelos (mismo orden que dev), no como cifra absoluta.
- **Leave-topic-out:** el AUC es estable por tema (0,88-0,94) pero el F1 varía (0,73-0,92) por calibración: sobrepredice Fake en Entertainment y Sport e infrapredice en Economy, Health y Security. Los temas pequeños (Education, Security, Economy, Health) son demasiado ruidosos para interpretarlos. Todo sigue siendo México 2018: no sustituye a una prueba de cambio de dominio.
- **Consecuencia:** el problema esperado en test no es la capacidad de ordenar sino el umbral y la calibración. Se comparan a continuación umbral base, umbral óptimo en dev y decisión por rango.

## Calibración y umbral

Se compara, para regresión logística y SVM (sin ComplementNB): umbral base, umbral óptimo en dev (calculado **solo con dev**) y decisión por rango. Test se usa únicamente para informar, no para elegir.

**Sobre la proporción del 50 % en la decisión por rango.** Es un supuesto, no un dato de las páginas oficiales de FakeDeS (la frase «quite balanced» de *Task & Data* se refiere al corpus de entrenamiento). Tiene tres apoyos: train (50,0 %) y dev (47,9 %) están balanceados; el test que usamos tiene 572 noticias, la cifra que dan varios artículos de participantes; el artículo de LCAD-UFES indica expresamente que el test tiene 286 noticias verdaderas y 286 falsas.

In [9]:
def compara_decisiones(nombre, kind, clf, col="input"):
    m = bl.make_pipe(kind, 1.0, clf).fit(tr[col], tr.label)
    s_dev, s_test = (bl.get_scores(m, d[col]) for d in (dev, test))
    base = 0.0 if clf == "svm" else 0.5
    grid = np.linspace(-1.0, 1.0, 201) if clf == "svm" else None   # el margen del SVM no está en [0, 1]
    t_dev = ev.best_threshold(dev.label, s_dev, grid)
    out = []
    for split, d, s in (("dev", dev, s_dev), ("test", test, s_test)):
        for dec, pred in ((f"umbral base ({base})", ev.predict_threshold(s, base)),
                          (f"óptimo dev ({t_dev:.2f})", ev.predict_threshold(s, t_dev)),
                          ("rango 50 %", ev.predict_by_rank(s, 0.5))):
            out.append({**ev.evaluate(d.label, pred, s, nombre), "split": split, "decision": dec})
    return out

cal = []
for nombre, kind, clf in (("char-lr", "char", "lr"), ("char-svm", "char", "svm"), ("word-lr", "word", "lr")):
    cal += compara_decisiones(nombre, kind, clf)
cal = pd.DataFrame(cal)
for sp in ("dev", "test"):
    print(sp.upper())
    display(cal[cal.split == sp][["modelo", "decision", "f1_fake", "precision", "recall", "pct_pred_fake", "auc"]]
            .round(3).reset_index(drop=True))

DEV


,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,char-lr,umbral base (0.5),0.782,0.766,0.799,0.500,0.868
1,char-lr,óptimo dev (0.47),0.800,0.735,0.878,0.572,0.868
2,char-lr,rango 50 %,0.782,0.766,0.799,0.500,0.868
3,char-svm,umbral base (0.0),0.806,0.792,0.820,0.497,0.886
4,char-svm,óptimo dev (0.01),0.814,0.809,0.820,0.486,0.886
5,char-svm,rango 50 %,0.803,0.786,0.820,0.500,0.886
6,word-lr,umbral base (0.5),0.771,0.766,0.777,0.486,0.873
7,word-lr,óptimo dev (0.48),0.797,0.752,0.849,0.541,0.873
8,word-lr,rango 50 %,0.782,0.766,0.799,0.500,0.873


TEST


,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,char-lr,umbral base (0.5),0.647,0.834,0.528,0.316,0.798
1,char-lr,óptimo dev (0.47),0.687,0.800,0.601,0.376,0.798
2,char-lr,rango 50 %,0.724,0.724,0.724,0.500,0.798
3,char-svm,umbral base (0.0),0.655,0.829,0.542,0.327,0.812
4,char-svm,óptimo dev (0.01),0.654,0.841,0.535,0.318,0.812
5,char-svm,rango 50 %,0.741,0.741,0.741,0.500,0.812
6,word-lr,umbral base (0.5),0.646,0.799,0.542,0.339,0.779
7,word-lr,óptimo dev (0.48),0.669,0.744,0.608,0.409,0.779
8,word-lr,rango 50 %,0.706,0.706,0.706,0.500,0.779


### Sensibilidad a la proporción asumida de Fake

Se repite la decisión por rango con otras proporciones (40-60 %) para ver cuánto depende el F1 del supuesto del 50 %. Es un análisis informativo: no se elige ninguna proporción con él.

In [10]:
m = bl.make_pipe("char", 1.0, "svm").fit(tr["input"], tr.label)
sens = []
for split, d in (("dev", dev), ("test", test)):
    s = bl.get_scores(m, d["input"])
    for frac in (0.40, 0.45, 0.50, 0.55, 0.60):
        sens.append({**ev.evaluate(d.label, ev.predict_by_rank(s, frac), s, f"char-svm rango {frac:.0%}"), "split": split})
sens = pd.DataFrame(sens)
display(sens[["split", "modelo", "f1_fake", "precision", "recall", "pct_pred_fake", "pct_real_fake"]].round(3))

,split,modelo,f1_fake,precision,recall,pct_pred_fake,pct_real_fake
0,dev,char-svm rango 40%,0.776,0.853,0.712,0.400,0.479
1,dev,char-svm rango 45%,0.796,0.823,0.770,0.448,0.479
2,dev,char-svm rango 50%,0.803,0.786,0.820,0.500,0.479
3,dev,char-svm rango 55%,0.803,0.750,0.863,0.552,0.479
4,dev,char-svm rango 60%,0.786,0.707,0.885,0.600,0.479
5,test,char-svm rango 40%,0.711,0.799,0.640,0.400,0.500
6,test,char-svm rango 45%,0.737,0.778,0.699,0.449,0.500
7,test,char-svm rango 50%,0.741,0.741,0.741,0.500,0.500
8,test,char-svm rango 55%,0.749,0.714,0.787,0.551,0.500
9,test,char-svm rango 60%,0.747,0.685,0.822,0.600,0.500


## Comparable con los baselines oficiales

Los organizadores publicaron tres baselines (char-3-gram SVM: 0,7062; BOW-SVM: 0,7217; BERT: 0,7321). Para compararnos en las mismas condiciones, se entrena con train + dev (como en el envío oficial) y se usa la decisión por defecto, sin rango. Se muestra también el rango al 50 % para ver cuánto aporta ese supuesto.

In [11]:
trdev = pd.concat([tr, dev], ignore_index=True)
oficial = []
for nombre, kind, clf in (("char-svm", "char", "svm"), ("word-svm", "word", "svm"),
                          ("char-lr", "char", "lr"), ("word-lr", "word", "lr")):
    m = bl.make_pipe(kind, 1.0, clf).fit(trdev["input"], trdev.label)
    s = bl.get_scores(m, test["input"])
    thr = 0.0 if clf == "svm" else 0.5
    for dec, pred in (("umbral por defecto", ev.predict_threshold(s, thr)), ("rango 50 %", ev.predict_by_rank(s, 0.5))):
        oficial.append({**ev.evaluate(test.label, pred, s, f"{nombre} (train+dev)"), "decision": dec})
oficial = ev.results_table(oficial)
display(oficial[["modelo", "decision", "f1_fake", "precision", "recall", "pct_pred_fake", "auc"]].reset_index(drop=True))

,modelo,decision,f1_fake,precision,recall,pct_pred_fake,auc
0,char-svm (train+dev),rango 50 %,0.745,0.745,0.745,0.500,0.816
1,char-lr (train+dev),rango 50 %,0.731,0.731,0.731,0.500,0.800
2,word-svm (train+dev),rango 50 %,0.727,0.727,0.727,0.500,0.799
3,word-lr (train+dev),rango 50 %,0.706,0.706,0.706,0.500,0.784
4,char-svm (train+dev),umbral por defecto,0.649,0.852,0.524,0.308,0.816
5,char-lr (train+dev),umbral por defecto,0.626,0.845,0.497,0.294,0.800
6,word-svm (train+dev),umbral por defecto,0.611,0.831,0.483,0.290,0.799
7,word-lr (train+dev),umbral por defecto,0.598,0.836,0.465,0.278,0.784


## Guardado y entrega

El modelo final se elige con dev y la validación cruzada, nunca con test. Se reentrena con train + dev y se escribe el fichero de entrega con la decisión por rango.

In [12]:
FINAL_KIND, FINAL_CLF = "char", "svm"   # mejor en dev y CV; cambiar solo con argumentos de dev/CV

res.to_csv(ROOT / "results" / "baselines" / "02_resultados.csv", index=False)
cal.to_csv(ROOT / "results" / "baselines" / "02_calibracion.csv", index=False)

trdev = pd.concat([tr, dev], ignore_index=True)
final = bl.make_pipe(FINAL_KIND, 1.0, FINAL_CLF).fit(trdev["input"], trdev.label)
s = bl.get_scores(final, test["input"])
pred = ev.predict_by_rank(s, 0.5)
print(ev.evaluate(test.label, pred, s, f"final {FINAL_KIND}-{FINAL_CLF} (train+dev, rango 50 %)"))
ev.write_submission(test["Id"], pred, ROOT / "results" / "baselines" / "submission_baseline.txt")

{'modelo': 'final char-svm (train+dev, rango 50 %)', 'f1_fake': 0.7447552447552448, 'precision': 0.7447552447552448, 'recall': 0.7447552447552448, 'pct_pred_fake': 0.5, 'pct_real_fake': 0.5, 'n': 572, 'auc': 0.816372438750061}


## Conclusiones y decisiones del baseline

**Resultados (F1 de la clase Fake, métrica oficial según la página *Evaluation* de FakeDeS; mejor sistema de la clasificación oficial: 0,7666).**
- En dev, los clasificadores lineales rondan 0,77-0,81 (AUC 0,86-0,89), con diferencias entre ellos dentro del ruido; el suelo es «todo Fake» (0,648). Char-svm es el mejor en dev y en validación cruzada.
- En test, el mejor baseline es char-svm con decisión por rango al 50 %: **0,741** (AUC 0,812); entrenado con train + dev, 0,745 (AUC 0,816). Ninguno supera el 0,7666 del primer clasificado (GDUFS_DM). En la clasificación oficial, 0,741-0,745 quedaría 4.º (tras 0,7666, 0,7548 y 0,7514), por encima de los baselines de los organizadores (BERT 0,7321; BOW-SVM 0,7217; char-3-gram SVM 0,7062) y del 0,7385 del 4.º clasificado oficial, pero **solo con la decisión por rango** (ver cautelas más abajo).

**Qué explica la caída de dev a test.**
- No es la capacidad de ordenar: el AUC pasa de 0,87-0,89 a 0,78-0,81, una caída moderada.
- Es sobre todo calibración. Con umbral base los modelos marcan solo el 32-34 % de test como Fake (F1 ≈ 0,65); con decisión por rango al 50 % el F1 sube a 0,72-0,74. Coincide con lo anticipado en el EDA.
- El umbral óptimo en dev **no se traslada**: mejora dev (0,782 → 0,800 en char-lr) pero apenas cambia en test (char-svm 0,655 → 0,654).

**Análisis de características.**
- Quitar el titular o no limpiar el texto no cambia el F1 ni el AUC con rango 50 % (char-svm: 0,741 y AUC ≈ 0,81 en los tres casos). Sí cambia la calibración con umbral: el texto sin limpiar predice más Fake (45 % frente a 33 %), lo que explica el 0,728 frente a 0,644 de la sección 11 del EDA: es un efecto de umbral, no de limpieza.
- ComplementNB es el más sensible a la regla de decisión (char-cnb con umbral: 0,243 en test, solo un 7,5 % de predicciones Fake) y no se usa para umbrales.

**Decisión de diseño y supuestos.**
- Regla adoptada: decisión por rango al 50 %, con char-svm entrenado en train + dev. Es un **supuesto**: el balance de test no está documentado en la web oficial. Con esa proporción, la precisión, el recall y el F1 coinciden porque la proporción predicha iguala la real.
- Elegido con dev y validación cruzada, no con test.
- Sensibilidad: con char-svm, variar la proporción asumida entre el 40 % y el 60 % mueve el F1 de test entre 0,711 y 0,749 (0,741 al 50 %). El resultado no depende de forma crítica del supuesto, aunque sí sube algo al asumir más Fake (55 %: 0,749).

**Contexto oficial confirmado (web de FakeDeS y papers de participantes).**
- **Métrica:** F1 de la clase Fake. Algunos artículos hablan de «macro-F1», pero la regla de *Evaluation* y la columna «F1 score» de la clasificación son el F1 sobre Fake.
- **Partición oficial:** train 676 (338/338), dev 295 (153 True, 142 Fake) y test 572. Aquí dev tiene 290 porque se eliminaron 4 duplicados train/dev y 1 interno. Los envíos oficiales se entrenaron con train + dev, igual que el modelo final.
- **Test oficial:** solo `Id`, `Headline` y `Text`; `Source`, `Link` y `Topic` del fichero etiquetado no estaban disponibles en la competición, lo que refuerza no usarlos.
- **Formato de entrega:** `"fakenews"<TAB>"Id"<TAB>"True|Fake"` con salto de línea; los `Id` de test van de 1 a 572. Se admiten dos *runs* (primario y secundario), que hay que marcar.

**Para las siguientes tareas.** Un modelo que no dependa del léxico de train (embeddings, transformer o LLM con evidencias) debería reducir la caída de AUC; la calibración se tratará con rango y con una proporción documentada. Prioridades: LLM con y sin recuperación de evidencias; BETO con cabeza y cola del texto y decisión por rango; y un ensemble con el baseline TF-IDF.